In [7]:
import numpy as np
import pandas as pd

In [8]:
data = pd.read_csv('./data/weather_dataset.csv')
data = data.drop(['id'], axis=1)
data

,outlook,temperature,humidity,wind,play
0,sunny,hot,high,weak,no
1,sunny,hot,high,strong,no
2,overcast,hot,high,weak,yes
3,rainy,mild,high,weak,yes
4,rainy,cool,normal,weak,yes
5,rainy,cool,normal,strong,no
6,overcast,cool,normal,strong,yes
7,sunny,mild,high,weak,no
8,sunny,cool,normal,weak,yes
9,rainy,mild,normal,weak,yes


In [9]:
class CART:
    def __init__(self):
        pass

    def gini_impurity(self, feature, target_col):
        elements, counts = np.unique(target_col, return_counts=True)
        gini_impurity = 1 - np.sum((counts[i] / np.sum(counts)) ** 2 for i in range(len(elements)))
        print(f"Gini Impurity for feature {feature}: {gini_impurity}")
        return gini_impurity

    def information_gain(self, data, feature, target_col):
        total_gini = self.gini_impurity(feature, data[target_col])
        values, counts = np.unique(data[feature], return_counts=True)

        gini_split = np.sum(
            (counts[i] / np.sum(counts)) * self.gini_impurity(data[data[feature] == values[i]], data[data[feature] == values[i]][target_col])
            for i in range(len(values)))
        print(f"feature: {feature}, total gini: {total_gini}, weighted gini: {gini_split}")
        print(f"Information Gain for feature {feature}: {total_gini - gini_split}")
        return total_gini - gini_split

    def cart(self, data, original_data, features, target, parent_node_class=None):
      # base case 1: if all class in the set are the same
      if len(np.unique(data[target])) == 1:
        print(f"returning with pure data")
        return np.unique(data[target])[0]

      # base case 2: if subset is empty -> return most frequent class in original dataset
      if len(data) == 0:
        print("returning with empty set")
        return np.unique(original_data[target])[
            np.argmax(np.unique(original_data[target], return_counts=True)[1])
        ]
      # base case 3: if feature set is empty -> return parent node class
      if len(features) == 0:
        print("returning with empty feature set")
        return parent_node_class

      # get most frequent class for fallback
      parent_node_class = np.unique(data[target])[np.argmax(np.unique(data[target], return_counts=True)[1])]
      gains = [self.information_gain(data, feature, target) for feature in features]
      best_feature = features[np.argmax(gains)]
      print(f"Best feature: {best_feature}")

      # initialize tree, create subset
      tree = {best_feature: {}}
      remaining_features = [f for f in features if f != best_feature]

      # build tree for each value in the best feature
      for value in np.unique(data[best_feature]):
        print(f"selected value: {value}")
        subset = data[data[best_feature] == value]
        subtree = self.cart(subset, original_data, remaining_features, target, parent_node_class)
        tree[best_feature][value] = subtree

      return tree

In [10]:
features = data.columns[:-1]
model = CART()
tree = model.cart(data, data, features, 'play')

print("\nDecision Tree:")
import pprint
pprint.pprint(tree)

Gini Impurity for feature outlook: 0.4591836734693877
Gini Impurity for feature      outlook temperature humidity    wind play
2   overcast         hot     high    weak  yes
6   overcast        cool   normal  strong  yes
11  overcast        mild     high  strong  yes
12  overcast         hot   normal    weak  yes: 0.0
Gini Impurity for feature    outlook temperature humidity    wind play
3    rainy        mild     high    weak  yes
4    rainy        cool   normal    weak  yes
5    rainy        cool   normal  strong   no
9    rainy        mild   normal    weak  yes
13   rainy        mild     high  strong   no: 0.48
Gini Impurity for feature    outlook temperature humidity    wind play
0    sunny         hot     high    weak   no
1    sunny         hot     high  strong   no
7    sunny        mild     high    weak   no
8    sunny        cool   normal    weak  yes
10   sunny        mild   normal  strong  yes: 0.48
feature: outlook, total gini: 0.4591836734693877, weighted gini: 0.342857142

C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\1121458911.py:7: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  gini_impurity = 1 - np.sum((counts[i] / np.sum(counts)) ** 2 for i in range(len(elements)))
C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\1121458911.py:15: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  gini_split = np.sum(
C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\1121458911.py:7: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  gini_impurity = 1 - np.sum((counts[i] / np.sum(counts)) ** 2 for i in range(len(elements)))
C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\